# Paper figures

Draws every figure of the paper from the two compact summaries in `results/summary/`
(no experiment needs to be rerun):

* `main_summary.pkl` — Figures 2, 3, 4, 6, 7, 9 (all 52 locations × 4 horizons, every learning rate)
* `sensitivity_summary.pkl` — Figures 5, 8 (US-agg runtime / grid-unit sweeps)

Both are produced by `src/summary.py` (see `run_experiments.ipynb`). Figures are written to `figures/`.

Conventions: *Total omni-error* and *Total QL (Rel.)* are reported relative to the best base
forecaster; quantile losses are divided by $Y_{\max}$ of each series. `*Our alg.` / `*Hedge-QL`
denote the best learning rate in hindsight.

In [ ]:
import os, sys, pickle
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir, "src")))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
import seaborn as sns

import config as cfg
from paper_plots import (RC_PARAMS, ENS_COLOR, ENS_STYLES, ENS_PLOTTED, lr_c_str_format,
                         log10_formatter, frac_steps_crossing, derived_metrics, quantile_plot,
                         standardized_ranks_from_losses, ridgeline_rank_violin)

plt.rcParams.update(RC_PARAMS)
FIG_DIR = cfg.FIG_DIR
os.makedirs(FIG_DIR, exist_ok=True)

S = pickle.load(open(cfg.MAIN_SUMMARY_PATH, "rb"))
SENS = pickle.load(open(cfg.SENSITIVITY_SUMMARY_PATH, "rb"))

WEEKS = S["weeks"]
US = S["geo_list"].index("US")          # states are indices [:US], US-agg is the last
DATES = pd.to_datetime(S["dates"])
ETA_O, ETA_H = S["eta_omni"], S["eta_ql"]

# The largest c_O (10^1.5) is excluded when searching for the best c_O of our algorithm.
D = derived_metrics(S, n_omni_eta=len(ETA_O) - 1)

def states(x):
    # (W, G, ...) -> (W * n_states, ...), week-major, US-agg dropped
    return x[:, :US].reshape(-1, *x.shape[2:])

## Figure 2

Omni-error over time on US-agg (`Omni_error_over_time.pdf`).

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 4.3), sharey="row")
i_c1 = list(ETA_O).index(1.0)
for wi, w in enumerate(WEEKS):
    a = ax[wi]
    bfst = S["bf_best_score"][wi, US]
    for _, eix, ename in ENS_PLOTTED:
        a.plot(DATES, S["ens_score"][wi, US, :, eix] - bfst, linewidth=2, color=ENS_COLOR,
               linestyle=ENS_STYLES[eix], label=ename)

    ih = S["ql_score"][wi, US, :, -1].argmin()
    a.plot(DATES, S["ql_score"][wi, US, ih] - bfst, color="forestgreen", linewidth=2,
           linestyle="--", label="*Hedge-QL")
    io = S["omni_score"][wi, US, :len(ETA_O) - 2, -1].argmin()
    a.plot(DATES, S["omni_score"][wi, US, io] - bfst, color="blue", linestyle="--",
           linewidth=2, label="*Our alg.")
    a.plot(DATES, S["omni_score"][wi, US, i_c1] - bfst, color="tab:red", linewidth=2,
           label="Our alg., " + lr_c_str_format(1.0, "o"))

    lr_legend = a.legend(
        handles=[Line2D([0], [0], color="green", linestyle="--", linewidth=2),
                 Line2D([0], [0], color="blue", linestyle="--", linewidth=2)],
        labels=["Hedge-QL " + lr_c_str_format(ETA_H[ih], "h"),
                "Our alg. " + lr_c_str_format(ETA_O[io], "o")],
        loc="upper right", fontsize=12)
    a.add_artist(lr_legend)

    a.axhline(0, color="red", linewidth=0.5, alpha=0.5, linestyle="-", label="y=0.0")
    a.set_ylim(-0.015, 0.07)
    a.set_xlabel("Date (yymm)", fontsize=14)
    a.tick_params(axis="x", labelrotation=45, labelsize=12)
    a.xaxis.set_major_formatter(mdates.DateFormatter("%y%m"))
    a.set_title(f"{w}wk-ahead prediction", fontsize=14)

ax[0].set_ylabel("Omni. error", fontsize=14)
ax[3].legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=13)
fig.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Omni_error_over_time.pdf"), bbox_inches="tight")
plt.show()

## Figure 3

**Left** (`Error_diff_density.pdf`): densities over the 51 state-level series of the total
omni-error / total QL of each method minus that of `*Our alg.`.
**Right** (`Difficulty_plot.pdf`): instantaneous QL of `*Our alg.` vs. `*Hedge-QL`, colored by
difficulty (average pinball loss of the base forecasters).

In [ ]:
w_colors = ["#2c7bb6", "#1a9850", "#f46d43", "#d73027"]
fig, ax = plt.subplots(3, 2, figsize=(7, 6), sharex="col")

for wi, w in enumerate(WEEKS):
    best_omni_omni = D["omni_score"][wi, :US, :, -1].min(axis=1)
    best_ql_omni = D["ql_score"][wi, :US, :, -1].min(axis=1)
    best_omni_ql = D["omni_ql"][wi, :US, :, -1].min(axis=1)
    best_ql_ql = D["ql_ql"][wi, :US, :, -1].min(axis=1)

    for axix, eix, ename in ENS_PLOTTED:
        sns.kdeplot(D["ens_score"][wi, :US, -1, eix] - best_omni_omni, fill=False,
                    label=f"{w}wk-ahead", ax=ax[axix, 0], color=w_colors[wi])
        sns.kdeplot(D["ens_ql"][wi, :US, eix, -1] - best_omni_ql, fill=False,
                    label=f"{w}wk-ahead", ax=ax[axix, 1], color=w_colors[wi])
        ax[axix, 0].set_title(ename, fontsize=15)
        ax[axix, 1].set_title(ename, fontsize=15)

    sns.kdeplot(best_ql_omni - best_omni_omni, fill=False, label=f"{w}wk-ahead", ax=ax[2, 0], color=w_colors[wi])
    sns.kdeplot(best_ql_ql - best_omni_ql, fill=False, label=f"{w}wk-ahead", ax=ax[2, 1], color=w_colors[wi])

for i in range(3):
    for j in range(2):
        ax[i, j].axvline(0, color="black", linestyle="-.")
        ax[i, j].set_yticks([])
        ax[i, j].set_ylabel("")
    ax[i, 0].set_ylabel("Density")
for j in range(2):
    ax[2, j].set_title("*Hedge-QL", fontsize=15)

ax[2, 0].set_xlabel("Omni. error rel. to *our alg.", fontsize=14)
ax[2, 1].set_xlabel("Quantile loss rel. to *our alg.", fontsize=14)
ax[2, 1].legend(fontsize=12)
fig.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Error_diff_density.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# per (series, t): best learning rate of each method in terms of QL up to t
best_lr_omni_ql = states(D["omni_ql"]).min(axis=1).flatten()
best_lr_ql_ql = states(D["ql_ql"]).min(axis=1).flatten()
# difficulty: average base-forecaster pinball at t, normalized by Y_t (rescaled by Y_max)
Y = S["Y"]
avg_f_difficulty = states(D["avg_f_t_ql"] / Y * Y.max(axis=-1, keepdims=True)).flatten()

tr_best_omni = np.sqrt(best_lr_omni_ql).clip(0.1, 0.25)
tr_best_ql = np.sqrt(best_lr_ql_ql).clip(0.1, 0.25)
tr_difficulty = np.sqrt(avg_f_difficulty).clip(0.2, 0.7)

colors = plt.get_cmap("coolwarm")(np.linspace(0, 1, 256))
colors[128 - 40:128 + 40] = "000000"     # widen the neutral band
new_cmap = LinearSegmentedColormap.from_list("custom", colors)

fig, ax = plt.subplots(figsize=(4, 3))
sc = ax.scatter(x=tr_best_omni, y=tr_best_ql, c=tr_difficulty, s=3, cmap=new_cmap)
fig.colorbar(sc, ax=ax).set_label("Difficulty")
xmin, xmax = ax.get_xlim(); ymin, ymax = ax.get_ylim()
low, high = min(xmin, ymin), max(xmax, ymax)
ax.plot([low, high], [low, high], linestyle="--", color="black", label="y=x")
ax.set_xlim(low, high); ax.set_ylim(low, high)
ax.legend()
ax.set_xticks([0.1, 0.15, 0.2, 0.25])
ax.set_yticks([0.1, 0.15, 0.2, 0.25])
ax.set_xlabel("Inst. QL of *Our alg.")
ax.set_ylabel("Inst. QL of *Hedge-QL")
plt.savefig(os.path.join(FIG_DIR, "Difficulty_plot.pdf"), bbox_inches="tight")
plt.show()

## Figure 4

Sensitivity of our algorithm to its learning rate $c_O \in \{10^{i/4}\}_{i=-8}^{5}$ on US-agg
(`Sensitivity.pdf`). Figure 7 below is the same plot for Hedge-QL.

In [ ]:
ETA_SLICE_O = slice(1, 14)    # c_O = 10^-2 ... 10^1.25
ETA_SLICE_H = slice(2, 16)    # c_H = 10^-1 ... 10^3

def ref_lines(ax, wi, star_label, star_score, star_ql, star_color):
    # ensembles + the best-in-hindsight competitor as horizontal lines (rows 0 and 1)
    bf_score, bf_ql = D["bf_best_score"][wi, US, -1], D["best_f_ql"][wi, US, -1]
    for _, eix, ename in ENS_PLOTTED:
        ax[0, wi].axhline(D["ens_score"][wi, US, -1, eix] - bf_score, label=ename,
                          linestyle=ENS_STYLES[eix], color=ENS_COLOR)
        ax[1, wi].axhline(D["ens_ql"][wi, US, eix, -1] - bf_ql, linestyle=ENS_STYLES[eix], color=ENS_COLOR)
    ax[0, wi].axhline(star_score - bf_score, label=star_label, linestyle="--", color=star_color, alpha=0.8)
    ax[1, wi].axhline(star_ql - bf_ql, linestyle="--", color=star_color, alpha=0.8)

fig, ax = plt.subplots(2, 4, figsize=(15, 5), sharex=True, sharey="row")
ax[0, 0].set_ylabel("Total omni-error")
ax[1, 0].set_ylabel("Total QL (Rel.)")
for wi, w in enumerate(WEEKS):
    ax[0, wi].set_title(f"{w}wk-ahead prediction")
    ax[1, wi].set_xlabel(r"Learning rate (log($c_O$))")
    ax[0, wi].set_xscale("log")
    for r in range(2):
        log10_formatter(ax[r, wi])
    ref_lines(ax, wi, "*Hedge-QL", D["ql_score"][wi, US, ETA_SLICE_H, -1].min(),
              D["ql_ql"][wi, US, ETA_SLICE_H, -1].min(), "green")
    ax[0, wi].plot(ETA_O[ETA_SLICE_O], D["omni_score"][wi, US, ETA_SLICE_O, -1] - D["bf_best_score"][wi, US, -1],
                   marker="o", label="Our algorithm", color="blue", alpha=0.8)
    ax[1, wi].plot(ETA_O[ETA_SLICE_O], D["omni_ql"][wi, US, ETA_SLICE_O, -1] - D["best_f_ql"][wi, US, -1],
                   marker="o", color="blue", alpha=0.8)

handles, labels = ax[0, 3].get_legend_handles_labels()
fig.tight_layout(h_pad=1.0)
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.savefig(os.path.join(FIG_DIR, "Sensitivity.pdf"), bbox_inches="tight")
plt.show()

## Figure 7

Sensitivity of Hedge-QL to its learning rate $c_H \in \{10^{i/4}\}_{i=-4}^{12}$ on US-agg
(`HQL_Sensitivity.pdf`), with the percentage of time steps at which Hedge-QL's quantiles cross
(our algorithm never crosses).

In [ ]:
hql_cross = np.array([[frac_steps_crossing(S["us_ql_preds"][wi, e]) for e in range(len(ETA_H))]
                      for wi in range(len(WEEKS))])
omni_cross = np.array([[frac_steps_crossing(S["us_omni_preds"][wi, e]) for e in range(len(ETA_O) - 1)]
                       for wi in range(len(WEEKS))])
print("max crossing frac -- Hedge-QL:", hql_cross.max(), " our alg.:", omni_cross.max())

fig, ax = plt.subplots(3, 4, figsize=(15, 7.5), sharex=True, sharey="row",
                       gridspec_kw={"height_ratios": [3, 3, 1.5]})
ax[0, 0].set_ylabel("Total omni-error")
ax[1, 0].set_ylabel("Total QL (Rel.)")
ax[2, 0].set_ylabel("Q. Crossing (%)")
for wi, w in enumerate(WEEKS):
    ax[0, wi].set_title(f"{w}wk-ahead prediction")
    ax[2, wi].set_xlabel(r"Learning rate (log($c_H$))")
    ax[0, wi].set_xscale("log")
    for r in range(3):
        log10_formatter(ax[r, wi])
    ref_lines(ax, wi, "*Our alg.", D["omni_score"][wi, US, 2:, -1].min(),
              D["omni_ql"][wi, US, 2:, -1].min(), "blue")
    ax[0, wi].plot(ETA_H[ETA_SLICE_H], D["ql_score"][wi, US, ETA_SLICE_H, -1] - D["bf_best_score"][wi, US, -1],
                   marker="o", label="Hedge-QL", color="green", alpha=0.8)
    ax[1, wi].plot(ETA_H[ETA_SLICE_H], D["ql_ql"][wi, US, ETA_SLICE_H, -1] - D["best_f_ql"][wi, US, -1],
                   marker="o", color="green", alpha=0.8)
    ax[2, wi].axhline(100 * omni_cross[wi].max(), linestyle="--", color="blue", alpha=0.8)
    ax[2, wi].plot(ETA_H[ETA_SLICE_H], 100 * hql_cross[wi, ETA_SLICE_H], marker="o", color="green", alpha=0.8)

handles, labels = ax[0, 3].get_legend_handles_labels()
fig.tight_layout(h_pad=1.0)
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.savefig(os.path.join(FIG_DIR, "HQL_Sensitivity.pdf"), bbox_inches="tight")
plt.show()

## Figure 5

Runtime of our algorithm ($c_O=1$, US-agg, 4-week-ahead) vs. grid unit, number of quantile levels
and number of base forecasters (`runtime_combined.pdf`). Absolute times depend on the machine.

In [ ]:
LABEL_FS, TICK_FS = 14, 12
g, q, f = SENS["runtime_grid"], SENS["runtime_quantile"], SENS["runtime_forecaster"]
g_units, g_times, g_ms = [r["unit"] for r in g], [r["runtime_sec"] for r in g], [r["m"] for r in g]

with plt.rc_context({k: plt.rcParamsDefault[k] for k in RC_PARAMS}):   # matplotlib default fonts
    fig, ax = plt.subplots(1, 3, figsize=(17, 4))

    ax[0].plot(g_units, g_times, "o-", color="tab:blue")
    ax[0].set_xscale("log"); ax[0].set_yscale("log")
    ax[0].set_xlim(15, 1400)
    ax[0].set_xticks([25, 50, 100, 250, 500, 1000])
    ax[0].set_yticks([1, 5, 10, 50, 100])
    for axis in (ax[0].xaxis, ax[0].yaxis):
        axis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
        axis.set_minor_formatter(plt.NullFormatter())
    for x, y, mm in zip(g_units, g_times, g_ms):
        ax[0].annotate(f"m={mm}", (x, y), textcoords="offset points", xytext=(-24, -18), fontsize=12.5)
    ax[0].invert_xaxis()
    ax[0].set_xlabel("Grid unit", fontsize=LABEL_FS)

    ax[1].plot([r["n_quantiles"] for r in q], [r["runtime_sec"] for r in q], "o-", color="tab:green")
    ax[1].set_xlabel("Number of quantile levels (N)", fontsize=LABEL_FS)

    ax[2].plot([r["n_forecasters"] for r in f], [r["runtime_sec"] for r in f], "o-", color="tab:purple")
    ax[2].set_xlabel("Number of base forecasters (B)", fontsize=LABEL_FS)

    for a in ax:
        a.set_ylabel("Runtime (s)", fontsize=LABEL_FS + 2)
        a.tick_params(axis="both", which="both", labelsize=TICK_FS + 2)
        a.grid(True, which="both", alpha=0.3)

    fig.tight_layout(w_pad=2)
    fig.savefig(os.path.join(FIG_DIR, "runtime_combined.pdf"), dpi=150)
    plt.show()

## Figure 6

Quantile forecasts on US-agg at 1- and 4-week-ahead horizons
(`Quantile_estimates_US_1wk.pdf`, `Quantile_estimates_US_4wk.pdf`). The learning rates of
`*Hedge-QL` / `*Our algorithm` are those with the best total omni-error (as in Figure 2).

In [ ]:
for w in [1, 4]:
    wi = WEEKS.index(w)
    unit = S["unit"][wi, US]
    y_orig = S["Y"][wi, US] * unit
    ih = S["ql_score"][wi, US, :, -1].argmin()
    io = S["omni_score"][wi, US, :len(ETA_O) - 2, -1].argmin()

    fig, ax = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
    for axix, eix, ename in ENS_PLOTTED:
        quantile_plot(ax[axix], DATES, S["us_ens_preds_orig"][wi, :, :, eix], y_orig, S["alpha_list"])
        ax[axix].set_title(ename, fontsize=14)
    quantile_plot(ax[2], DATES, S["us_ql_preds"][wi, ih] * unit, y_orig, S["alpha_list"])
    quantile_plot(ax[3], DATES, S["us_omni_preds"][wi, io] * unit, y_orig, S["alpha_list"])
    ax[2].set_title(f'*Hedge-QL ({lr_c_str_format(ETA_H[ih], "h")})', fontsize=14)
    ax[3].set_title(f'*Our algorithm ({lr_c_str_format(ETA_O[io], "o")})', fontsize=14)

    ax[0].set_ylabel("Weekly hosp.", fontsize=12)
    for a in ax:
        a.set_xlabel("Date (yymm)", fontsize=12)
        a.set_ylim(0, 400000)
        a.legend(fontsize=13)
    fig.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"Quantile_estimates_US_{w}wk.pdf"), bbox_inches="tight")
    plt.show()

## Figure 8

Total QL (Rel.) of our algorithm at fixed $c_O$ and of `*Hedge-QL` vs. grid unit on US-agg
(`Sensitivity_grid_QL.pdf`). Ours uses the expected pinball loss of its randomized prediction.

In [ ]:
C_O_FIXED = [(0, "o", "tab:purple"), (0.5, "d", "tab:blue"), (1, "s", "tab:orange")]   # (log10 c_O, marker, color)

with plt.rc_context({k: plt.rcParamsDefault[k] for k in RC_PARAMS}):   # matplotlib default fonts
    fig, ax = plt.subplots(1, len(SENS["grid_ql"]), figsize=(14, 3.5))
    ax[0].set_ylabel(r"Total QL (Rel.) $\times 10^4$", fontsize=14)
    for a, (w, r) in zip(ax, sorted(SENS["grid_ql"].items())):
        rel = lambda v: (np.asarray(v) - r["best_forecaster_mean_ql"]) / r["y_max"] * 1e4
        a.plot(r["units"], rel(r["ql_mean_ql"].min(axis=1)), "--", marker="^", color="green",
               alpha=0.8, label="*Hedge-QL")
        for c_o, mk, col in C_O_FIXED:
            ie = r["eta_omni"].index(round(10 ** c_o, 4))
            a.plot(r["units"], rel(r["omni_mean_ql"][:, ie]), marker=mk, color=col, alpha=0.8,
                   label=rf"Our alg., $c_O=10^{{{c_o}}}$")
        a.set_title(f"{w}wk-ahead prediction")
        a.set_xlabel("Grid unit", fontsize=14)
        a.set_xscale("log")
        a.set_xticks([25, 50, 100, 250, 500, 1000])
        a.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
        a.xaxis.set_minor_formatter(plt.NullFormatter())
        a.invert_xaxis()
        a.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax[3].legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=11)
    fig.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "Sensitivity_grid_QL.pdf"), bbox_inches="tight")
    plt.show()

## Figure 9

Distribution of standardized ranks of instantaneous QL over all (state, horizon, time) triples
(`QL_std_rank.pdf`). Base forecasters are ranked on their raw (non-imputed) forecasts.

In [ ]:
OMNI_ETA_IDX = [8, 9, 10, 11, 12]     # c_O = 10^0 ... 10^1
QL_ETA_IDX = [13, 14]                 # c_H = 10^2, 10^2.5

cols = [states(D["omni_t_ql"][:, :, i]).flatten() for i in OMNI_ETA_IDX]
cols += [states(D["ql_t_ql"][:, :, i]).flatten() for i in QL_ETA_IDX]
cols += [states(D["omni_t_ql"]).min(axis=1).flatten(), states(D["ql_t_ql"]).min(axis=1).flatten()]
ens = states(D["ens_t_ql"][:, :, [0, 2]])                         # (n, 2, T)
ranks_mtx = np.column_stack(cols + [ens.transpose(0, 2, 1).reshape(-1, 2),
                                    states(D["bf_raw_t_ql"]).reshape(-1, D["bf_raw_t_ql"].shape[-1])])

names = ([r"Our alg., lr=$10^0$", r"Our alg., lr=$10^{0.25}$", r"Our alg., lr=$10^{0.5}$",
          r"Our alg., lr=$10^{0.75}$", r"Our alg., lr=$10^1$", r"Hedge-QL, lr=$10^2$",
          r"Hedge-QL, lr=$10^{2.5}$", "*Our alg.", "*Hedge-QL", "COVIDhub_Ens", "JHUAPL-SLPEns"]
         + S["raw_forecaster_names"])
rank_df = standardized_ranks_from_losses(pd.DataFrame(ranks_mtx, columns=names), lower_is_better=True)
order = rank_df.quantile(0.25, axis=0).sort_values()[::-1].index.tolist()

fig, ax = ridgeline_rank_violin(rank_df, model_order=order, bandwidth=0.20, width=0.42, figsize=(7, 6))
plt.savefig(os.path.join(FIG_DIR, "QL_std_rank.pdf"), bbox_inches="tight")
plt.show()